In [1]:
"""
Compare Holm vs Benjamini-Hochberg (FDR) correction on the Q1 monthly
Granger causality results.

Reads the CSV already produced by granger_q1_monthly_v3.ipynb (which has
p_value and the Holm-based p_value_adj/reject_adj), adds BH-adjusted
columns alongside for a direct comparison, and prints a summary.

Does not overwrite your existing padjusted CSV -- saves a new file with
"_holm_vs_bh" appended so both versions exist side by side.
"""

import pandas as pd
import pingouin as pg
import matplotlib.pyplot as plt
import os

# ----------------------------------------------------------------------
# EDIT THIS PATH to point at your actual Q1 monthly results file
# ----------------------------------------------------------------------
IN_PATH = "/Users/nadia/Desktop/redditRun_june/q1_analysis_v2/q1_granger_causality_results_v3(monthly)_padjusted.csv"

ALPHA = 0.05


def plot_comparison(df_sorted, out_path):
    """Horizontal grouped bar chart: raw p-value vs Holm-adjusted vs
    BH-adjusted, one row per test, sorted ascending by raw p-value
    (same order Holm's step-down procedure uses). A dashed red line marks
    alpha=0.05 so it's visually obvious which bars cross the threshold."""
    label_cols = [c for c in ["bat_threshold", "cve_threshold", "criterion", "lag"] if c in df_sorted.columns]
    if label_cols:
        labels = df_sorted[label_cols].astype(str).agg(" | ".join, axis=1)
    else:
        labels = df_sorted.index.astype(str)

    n = len(df_sorted)
    y = range(n)
    bar_h = 0.25

    fig, ax = plt.subplots(figsize=(9, max(3, n * 0.5)))

    ax.barh([i + bar_h for i in y], df_sorted["p_value"], height=bar_h,
            color="#898781", label="Raw p-value")
    ax.barh([i for i in y], df_sorted["p_value_holm"], height=bar_h,
            color="#2a78d6", label="Holm-adjusted")
    ax.barh([i - bar_h for i in y], df_sorted["p_value_bh"], height=bar_h,
            color="#1baf7a", label="BH (FDR)-adjusted")

    ax.axvline(ALPHA, color="#e34948", linestyle="--", linewidth=1.5,
               label=f"alpha = {ALPHA}")

    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()  # smallest p-value (most significant) at the top
    ax.set_xlim(0, 1)
    ax.set_xlabel("p-value")
    ax.set_title("Raw vs Holm vs BH-adjusted p-values (Q1 monthly, cve_to_bat)")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()

    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Saved plot -> {out_path}")


def main():
    if not os.path.exists(IN_PATH):
        print(f"Could not find {IN_PATH}. Edit IN_PATH at the top of this script.")
        return

    df = pd.read_csv(IN_PATH)
    print(f"Loaded {len(df)} rows from {IN_PATH}")

    if "p_value" not in df.columns:
        raise ValueError(f"No 'p_value' column found. Columns present: {list(df.columns)}")

    # Recompute Holm here too, so both methods are computed fresh and
    # comparably (in case the existing p_value_adj was rounded differently)
    reject_holm, p_holm = pg.multicomp(df["p_value"].values, alpha=ALPHA, method="holm")
    reject_bh, p_bh = pg.multicomp(df["p_value"].values, alpha=ALPHA, method="fdr_bh")

    df["p_value_holm"] = p_holm
    df["reject_holm"] = reject_holm
    df["p_value_bh"] = p_bh
    df["reject_bh"] = reject_bh

    # Sort by raw p-value ascending, same order Holm's step-down uses
    df_sorted = df.sort_values("p_value").reset_index(drop=True)

    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", None)
    print("\n" + "=" * 70)
    print("FULL COMPARISON (sorted by raw p-value)")
    print("=" * 70)
    cols_to_show = [c for c in [
        "bat_threshold", "cve_threshold", "criterion", "lag",
        "p_value", "p_value_holm", "reject_holm", "p_value_bh", "reject_bh"
    ] if c in df_sorted.columns]
    print(df_sorted[cols_to_show].to_string(index=False))

    n = len(df)
    n_raw_sig = int((df["p_value"] < ALPHA).sum())
    n_holm_sig = int(reject_holm.sum())
    n_bh_sig = int(reject_bh.sum())

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Total tests: {n}")
    print(f"Significant BEFORE any correction: {n_raw_sig}/{n}")
    print(f"Significant after Holm (FWER):      {n_holm_sig}/{n}")
    print(f"Significant after BH (FDR):          {n_bh_sig}/{n}")

    if n_holm_sig != n_bh_sig:
        print("\nHolm and BH disagree on the significance call for at least one test --")
        print("inspect the rows above where reject_holm and reject_bh differ.")
    else:
        print("\nHolm and BH agree on every significance call. BH is less conservative")
        print("in general (smaller adjusted p-values for mid-ranked tests), but here")
        print("it doesn't change which tests cross the alpha=0.05 line.")

    out_path = IN_PATH.replace(".csv", "_holm_vs_bh.csv")
    df_sorted.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")

    plot_path = IN_PATH.replace(".csv", "_holm_vs_bh.png")
    plot_comparison(df_sorted, plot_path)


if __name__ == "__main__":
    main()

Loaded 12 rows from /Users/nadia/Desktop/redditRun_june/q1_analysis_v2/q1_granger_causality_results_v3(monthly)_padjusted.csv

FULL COMPARISON (sorted by raw p-value)
bat_threshold cve_threshold criterion  lag  p_value  p_value_holm  reject_holm  p_value_bh  reject_bh
    n_bat_ge2    n_cve_run2       AIC    4 0.003568      0.042816         True    0.042816       True
    n_bat_ge1    n_cve_run2       AIC    4 0.010098      0.111078        False    0.060588      False
    n_bat_ge2    n_cve_run2       BIC    1 0.123609      1.000000        False    0.494436      False
    n_bat_ge4    n_cve_run2       AIC    7 0.301886      1.000000        False    0.713231      False
    n_bat_ge1    n_cve_run2       BIC    1 0.404676      1.000000        False    0.713231      False
    n_bat_ge2    n_cve_run1       BIC    1 0.486824      1.000000        False    0.713231      False
    n_bat_ge1    n_cve_run1       AIC    2 0.516460      1.000000        False    0.713231      False
    n_bat_ge1    

In [2]:
"""
Compare Holm vs Benjamini-Hochberg (FDR) correction on the Q1 monthly
Granger causality results.

Reads the CSV already produced by granger_q1_monthly_v3.ipynb (which has
p_value and the Holm-based p_value_adj/reject_adj), adds BH-adjusted
columns alongside for a direct comparison, and prints a summary.

Does not overwrite your existing padjusted CSV -- saves a new file with
"_holm_vs_bh" appended so both versions exist side by side.
"""

import pandas as pd
import pingouin as pg
import matplotlib.pyplot as plt
import os

# ----------------------------------------------------------------------
# EDIT THIS PATH to point at your actual Q1 monthly results file
# ----------------------------------------------------------------------
IN_PATH = "/Users/nadia/Desktop/redditRun_june/q2_analysis_v2/exit_granger_causality_results(monthly)_padjusted.csv"

ALPHA = 0.05


def plot_comparison(df_sorted, out_path):
    """Horizontal grouped bar chart: raw p-value vs Holm-adjusted vs
    BH-adjusted, one row per test, sorted ascending by raw p-value
    (same order Holm's step-down procedure uses). A dashed red line marks
    alpha=0.05 so it's visually obvious which bars cross the threshold."""
    label_cols = [c for c in ["bat_threshold", "exit_threshold", "cve_threshold", "criterion", "lag"]
                  if c in df_sorted.columns]
    if label_cols:
        labels = df_sorted[label_cols].astype(str).agg(" | ".join, axis=1)
    else:
        labels = df_sorted.index.astype(str)

    n = len(df_sorted)
    y = range(n)
    bar_h = 0.25

    fig, ax = plt.subplots(figsize=(9, max(3, n * 0.5)))

    ax.barh([i + bar_h for i in y], df_sorted["p_value"], height=bar_h,
            color="#898781", label="Raw p-value")
    ax.barh([i for i in y], df_sorted["p_value_holm"], height=bar_h,
            color="#2a78d6", label="Holm-adjusted")
    ax.barh([i - bar_h for i in y], df_sorted["p_value_bh"], height=bar_h,
            color="#1baf7a", label="BH (FDR)-adjusted")

    ax.axvline(ALPHA, color="#e34948", linestyle="--", linewidth=1.5,
               label=f"alpha = {ALPHA}")

    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()  # smallest p-value (most significant) at the top
    ax.set_xlim(0, 1)
    ax.set_xlabel("p-value")
    direction = df_sorted["direction"].iloc[0] if "direction" in df_sorted.columns else "cve_to_bat"
    ax.set_title(f"Raw vs Holm vs BH-adjusted p-values (monthly, {direction})")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()

    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Saved plot -> {out_path}")


def main():
    if not os.path.exists(IN_PATH):
        print(f"Could not find {IN_PATH}. Edit IN_PATH at the top of this script.")
        return

    df = pd.read_csv(IN_PATH)
    print(f"Loaded {len(df)} rows from {IN_PATH}")

    if "p_value" not in df.columns:
        raise ValueError(f"No 'p_value' column found. Columns present: {list(df.columns)}")

    # Recompute Holm here too, so both methods are computed fresh and
    # comparably (in case the existing p_value_adj was rounded differently)
    reject_holm, p_holm = pg.multicomp(df["p_value"].values, alpha=ALPHA, method="holm")
    reject_bh, p_bh = pg.multicomp(df["p_value"].values, alpha=ALPHA, method="fdr_bh")

    df["p_value_holm"] = p_holm
    df["reject_holm"] = reject_holm
    df["p_value_bh"] = p_bh
    df["reject_bh"] = reject_bh

    # Sort by raw p-value ascending, same order Holm's step-down uses
    df_sorted = df.sort_values("p_value").reset_index(drop=True)

    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", None)
    print("\n" + "=" * 70)
    print("FULL COMPARISON (sorted by raw p-value)")
    print("=" * 70)
    cols_to_show = [c for c in [
        "bat_threshold", "exit_threshold", "cve_threshold", "criterion", "lag",
        "p_value", "p_value_holm", "reject_holm", "p_value_bh", "reject_bh"
    ] if c in df_sorted.columns]
    print(df_sorted[cols_to_show].to_string(index=False))

    n = len(df)
    n_raw_sig = int((df["p_value"] < ALPHA).sum())
    n_holm_sig = int(reject_holm.sum())
    n_bh_sig = int(reject_bh.sum())

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Total tests: {n}")
    print(f"Significant BEFORE any correction: {n_raw_sig}/{n}")
    print(f"Significant after Holm (FWER):      {n_holm_sig}/{n}")
    print(f"Significant after BH (FDR):          {n_bh_sig}/{n}")

    if n_holm_sig != n_bh_sig:
        print("\nHolm and BH disagree on the significance call for at least one test --")
        print("inspect the rows above where reject_holm and reject_bh differ.")
    else:
        print("\nHolm and BH agree on every significance call. BH is less conservative")
        print("in general (smaller adjusted p-values for mid-ranked tests), but here")
        print("it doesn't change which tests cross the alpha=0.05 line.")

    out_path = IN_PATH.replace(".csv", "_holm_vs_bh.csv")
    df_sorted.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")

    plot_path = IN_PATH.replace(".csv", "_holm_vs_bh.png")
    plot_comparison(df_sorted, plot_path)


if __name__ == "__main__":
    main()

Loaded 8 rows from /Users/nadia/Desktop/redditRun_june/q2_analysis_v2/exit_granger_causality_results(monthly)_padjusted.csv

FULL COMPARISON (sorted by raw p-value)
 exit_threshold cve_threshold criterion  lag  p_value  p_value_holm  reject_holm  p_value_bh  reject_bh
n_exit_explicit    n_cve_run2       AIC    5 0.008317      0.066536        False    0.066536      False
     n_exit_any    n_cve_run1       BIC    1 0.037348      0.261436        False    0.090128      False
     n_exit_any    n_cve_run2       AIC    3 0.038824      0.261436        False    0.090128      False
n_exit_explicit    n_cve_run2       BIC    2 0.045064      0.261436        False    0.090128      False
     n_exit_any    n_cve_run1       AIC    2 0.115832      0.463328        False    0.185331      False
     n_exit_any    n_cve_run2       BIC    1 0.320398      0.961194        False    0.427197      False
n_exit_explicit    n_cve_run1       BIC    1 0.463233      0.961194        False    0.529409      False
n_e